In [12]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

In [87]:
X_train_encoded = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_train_encoded.parquet")
X_valid_encoded = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_valid_encoded.parquet")
X_predict_encoded = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\X_predict_encoded.parquet")
y_train = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_train.parquet")
y_valid = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_valid.parquet")
y_predict = pd.read_parquet(r"C:\Users\Quang\Documents\Code\Self Project\PL Prediction\dataset\y_predict.parquet")

In [5]:
X_tv = pd.concat([X_train_encoded, X_valid_encoded], ignore_index=True).sort_values('Date', ascending=True)
y_tv = pd.concat([y_train, y_valid], ignore_index=True).sort_values('Date', ascending=True)

In [43]:
y_tv_home = y_tv["FTHG"]
y_tv_away = y_tv["FTAG"]
xgb_home = XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=67,
    n_jobs=-1
)
xgb_away = XGBRegressor(
    n_estimators=500,
    learning_rate=0.01,
    max_depth=3,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=1,
    random_state=67,
    n_jobs=-1
)
xgb_home.fit(X_tv.drop(columns=['Date']), y_tv_home.drop(columns=['Date']))
xgb_away.fit(X_tv.drop(columns=['Date']), y_tv_away.drop(columns=['Date']))

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,1
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [79]:
def feature_enginnering(df, matchday):

    df_history = pd.concat([df.dropna(subset=['FTHG']), df[df['Matchday'] == matchday]], ignore_index=True).sort_values('Date', ascending=True)
    df_home_history = df_history[['Date', 'HomeTeam', 'FTHG', 'FTAG', 'HomePoint']].rename(columns={'HomeTeam': 'Team', 'FTHG': 'GoalsScored', 'FTAG': 'GoalsConceded', 'HomePoint': 'Point'}).sort_values(['Team', 'Date']).reset_index(drop=True)
    df_home_history['Side'] = 'Home'
    df_away_history = df_history[['Date', 'AwayTeam', 'FTAG', 'FTHG', 'AwayPoint']].rename(columns={'AwayTeam': 'Team', 'FTAG': 'GoalsScored', 'FTHG': 'GoalsConceded', 'AwayPoint': 'Point'}).sort_values(['Team', 'Date']).reset_index(drop=True)
    df_away_history['Side'] = 'Away'
    df_team_history = pd.concat([df_home_history, df_away_history], ignore_index=True).sort_values(['Team', 'Date']).reset_index(drop=True)

    for col, new_col in [('GoalsScored', 'goals_avg_last_5'),('GoalsConceded', 'conceded_avg_last_5'),('Point', 'points_avg_last_5'),]:
        df_team_history[new_col] = (df_team_history.groupby('Team')[col].transform(lambda x: x.shift(1).rolling(5).mean()))
    for col, new_col in [('GoalsScored', 'home_goals_avg_home_last_5'),('GoalsConceded', 'home_conceded_avg_home_last_5')]:
        df_home_history[new_col] = (df_home_history.groupby('Team')[col].transform(lambda x: x.shift(1).rolling(5).mean()))
    for col, new_col in [('GoalsScored', 'away_goals_avg_away_last_5'),('GoalsConceded', 'away_conceded_avg_away_last_5')]:
        df_away_history[new_col] = (df_away_history.groupby('Team')[col].transform(lambda x: x.shift(1).rolling(5).mean()))

    df_team_history_home = df_team_history[df_team_history['Side'] == 'Home'][['Date', 'Team', 'goals_avg_last_5', 'conceded_avg_last_5', 'points_avg_last_5']].rename(columns={
            'Team': 'HomeTeam',
            'goals_avg_last_5': 'home_goals_avg_last_5',
            'conceded_avg_last_5': 'home_conceded_avg_last_5',
            'points_avg_last_5': 'home_points_avg_last_5'
        })
    df_team_history_away = df_team_history[df_team_history['Side'] == 'Away'][['Date', 'Team', 'goals_avg_last_5', 'conceded_avg_last_5', 'points_avg_last_5']].rename(columns={
            'Team': 'AwayTeam',
            'goals_avg_last_5': 'away_goals_avg_last_5',
            'conceded_avg_last_5': 'away_conceded_avg_last_5',
            'points_avg_last_5': 'away_points_avg_last_5'
        })
    df_home_history = df_home_history[['Date', 'Team', 'home_goals_avg_home_last_5', 'home_conceded_avg_home_last_5']].rename(columns={'Team': 'HomeTeam'})
    df_away_history = df_away_history[['Date', 'Team', 'away_goals_avg_away_last_5', 'away_conceded_avg_away_last_5']].rename(columns={'Team': 'AwayTeam'})

    df = df.drop(columns=df_team_history_home.columns.difference(['Date', 'HomeTeam'])).merge(df_team_history_home, on=['Date', 'HomeTeam'], how='left', validate='many_to_one')
    df = df.drop(columns=df_team_history_away.columns.difference(['Date', 'AwayTeam'])).merge(df_team_history_away, on=['Date', 'AwayTeam'], how='left', validate='many_to_one')
    df = df.drop(columns=df_home_history.columns.difference(['Date', 'HomeTeam'])).merge(df_home_history, on=['Date', 'HomeTeam'], how='left', validate='many_to_one')
    df = df.drop(columns=df_away_history.columns.difference(['Date', 'AwayTeam'])).merge(df_away_history, on=['Date', 'AwayTeam'], how='left', validate='many_to_one')

    return df

In [81]:
fe = feature_enginnering(X_predict_encoded, 1)
FTHG_pred = xgb_home.predict(fe[fe['Matchday'] == 1][xgb_home.get_booster().feature_names])
FTAG_pred = xgb_away.predict(fe[fe['Matchday'] == 1][xgb_away.get_booster().feature_names])

current_matches = X_predict_encoded[X_predict_encoded["Matchday"] == (1)]
current_matches['FTHG'] = FTHG_pred.round().astype('int8')
current_matches['FTAG'] = FTAG_pred.round().astype('int8')
FTR = [
    current_matches['FTHG'] > current_matches['FTAG'],
    current_matches['FTHG'] == current_matches['FTAG'],
    current_matches['FTHG'] < current_matches['FTAG']
]
home_point = [3, 1, 0]
away_point = [0, 1, 3]
current_matches['HomePoint'] = np.select(FTR, home_point, default=np.nan)
current_matches['AwayPoint'] = np.select(FTR, away_point, default=np.nan)

temp1 = X_predict_encoded.set_index(['Date', 'Matchday', 'HomeTeam', 'AwayTeam'])
temp2 = current_matches[['Date', 'Matchday', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'HomePoint', 'AwayPoint']].set_index(['Date', 'Matchday', 'HomeTeam', 'AwayTeam'])
temp1.update(temp2)
temp3 = temp1.reset_index()

In [84]:
X_predict_encoded.columns

Index(['Date', 'Matchday', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'HomePoint',
       'AwayPoint', 'home_goals_avg_last_5', 'home_conceded_avg_last_5',
       'home_points_avg_last_5', 'away_goals_avg_last_5',
       'away_conceded_avg_last_5', 'away_points_avg_last_5',
       'home_goals_avg_home_last_5', 'home_conceded_avg_home_last_5',
       'away_goals_avg_away_last_5', 'away_conceded_avg_away_last_5',
       'HomeTeam_Arsenal', 'HomeTeam_Aston Villa', 'HomeTeam_Blackburn',
       'HomeTeam_Bolton', 'HomeTeam_Bournemouth', 'HomeTeam_Brentford',
       'HomeTeam_Brighton', 'HomeTeam_Burnley', 'HomeTeam_Cardiff',
       'HomeTeam_Chelsea', 'HomeTeam_Crystal Palace', 'HomeTeam_Everton',
       'HomeTeam_Fulham', 'HomeTeam_Huddersfield', 'HomeTeam_Hull',
       'HomeTeam_Ipswich', 'HomeTeam_Leeds', 'HomeTeam_Leicester',
       'HomeTeam_Liverpool', 'HomeTeam_Luton', 'HomeTeam_Man City',
       'HomeTeam_Man United', 'HomeTeam_Middlesbrough', 'HomeTeam_Newcastle',
       'HomeTeam_Nor

In [88]:
for matchday in range(38):
    fe = feature_enginnering(X_predict_encoded, (matchday + 1))
    FTHG_pred = xgb_home.predict(fe[fe['Matchday'] == (matchday + 1)][xgb_home.get_booster().feature_names])
    FTAG_pred = xgb_away.predict(fe[fe['Matchday'] == (matchday + 1)][xgb_away.get_booster().feature_names])

    current_matches = X_predict_encoded[X_predict_encoded["Matchday"] == (matchday + 1)].copy()
    current_matches['FTHG'] = FTHG_pred.round().astype('int8')
    current_matches['FTAG'] = FTAG_pred.round().astype('int8')
    FTR = [
        current_matches['FTHG'] > current_matches['FTAG'],
        current_matches['FTHG'] == current_matches['FTAG'],
        current_matches['FTHG'] < current_matches['FTAG']
    ]
    home_point = [3, 1, 0]
    away_point = [0, 1, 3]
    current_matches['HomePoint'] = np.select(FTR, home_point, default=np.nan)
    current_matches['AwayPoint'] = np.select(FTR, away_point, default=np.nan)

    temp1 = X_predict_encoded.set_index(['Date', 'Matchday', 'HomeTeam', 'AwayTeam'])
    temp2 = current_matches[['Date', 'Matchday', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'HomePoint', 'AwayPoint']].set_index(['Date', 'Matchday', 'HomeTeam', 'AwayTeam'])
    temp1.update(temp2)
    X_predict_encoded  = temp1.reset_index()

In [89]:
final = X_predict_encoded.dropna(subset=['Matchday'])[['Date', 'Matchday', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'HomePoint','AwayPoint']]

In [ ]:
final


In [95]:
final_home = final[['Date', 'HomeTeam', 'FTHG', 'HomePoint']].rename(columns={'HomeTeam': 'Team', 'FTHG': 'Goal', 'HomePoint': 'Point'}).sort_values(['Team', 'Date']).reset_index(drop=True)
final_away = final[['Date', 'AwayTeam', 'FTAG', 'HomePoint']].rename(columns={'AwayTeam': 'Team', 'FTAG': 'Goal', 'HomePoint': 'Point'}).sort_values(['Team', 'Date']).reset_index(drop=True)
final_home['Side'] = 'Home'
final_away['Side'] = 'Away'
final2 = pd.concat([final_home, final_away], ignore_index=True).sort_values(['Date']).reset_index(drop=True)

In [96]:
final2

,Date,Team,Goal,Point,Side
0,2026-08-21,Arsenal,2.0,3.0,Home
1,2026-08-21,Coventry City,1.0,3.0,Away
2,2026-08-22,Crystal Palace,1.0,3.0,Away
3,2026-08-22,Brentford,2.0,3.0,Home
4,2026-08-22,Everton,2.0,3.0,Home
...,...,...,...,...,...
755,2027-05-30,Fulham,1.0,3.0,Away
756,2027-05-30,Leeds,1.0,1.0,Away
757,2027-05-30,Man City,2.0,0.0,Away
758,2027-05-30,Ipswich,1.0,0.0,Home


In [99]:
final2.groupby('Team')[['Point', 'Goal']].sum().sort_values(by='Point', ascending=False)

,Point,Goal
Team,,
Tottenham,79.0,64.0
Arsenal,75.0,66.0
Fulham,75.0,51.0
Chelsea,74.0,68.0
Bournemouth,72.0,50.0
Man United,72.0,58.0
Liverpool,70.0,73.0
Brentford,65.0,45.0
Aston Villa,63.0,46.0
